In [12]:
import os
import json
import MeCab
import shutil
import pyarrow as pa
import pyarrow.parquet as pq
import subprocess
import ahocorasick
from tqdm import tqdm
from datetime import datetime

In [10]:
# データベースの保存先のディレクトリ
TARGET_IMAGE_DIR = "BoketeDBv3/images"
TARGET_BOKE_DIR = "BoketeDBv3/boke_db"
TARGET_ODAI_DIR = "BoketeDBv3/odai_db"

# 元データのディレクトリ
IMAGE_DIR = "Bokete_Scraping_v3.1/images"
ODAI_DIR = "Bokete_Scraping_v3.1/odai"
BOKE_DIR = "Bokete_Scraping_v3.1/boke"
CAPTION_DIR = "Bokete_Scraping_v3.1/image_captions"
OCR_DIR = "Bokete_Scraping_v3.1/image_ocr"
TYPE_DIR = "Bokete_Scraping_v3.1/image_type"
RATING_DIR = "Bokete_Scraping_v3.1/rating"

In [15]:
if os.path.exists("BoketeDBv3"):
    subprocess.run(["mkdir", "empty_dir"])
    subprocess.run(["rsync", "-av", "--delete", "empty_dir/", "BoketeDBv3/"])
    subprocess.run(["rm", "-rf", "empty_dir"])
    
target_dirs = [
    TARGET_IMAGE_DIR,
    TARGET_BOKE_DIR,
    TARGET_ODAI_DIR
]
for D in target_dirs:
    os.makedirs(D, exist_ok=True)

sending incremental file list
./

sent 56 bytes  received 26 bytes  164.00 bytes/sec
total size is 0  speedup is 0.00


In [16]:
class WordChecker:
    def __init__(self, vocab):  # アンダーバーを左右2本ずつに修正
        self.auto = ahocorasick.Automaton()
        for index, V in enumerate(vocab):
            self.auto.add_word(V, (index, V))
        self.auto.make_automaton()
    
    def __call__(self, sentence):
        for _ in self.auto.iter(sentence):
            return True
        return False

with open("NG_words.json", "r", encoding="utf-8") as f:
    ng_words = json.load(f)
wc = WordChecker(ng_words)

class UniqueNounChecker:
    def __init__(self):
        self.seen_nouns = set()
        os.environ["MECABRC"] = '/etc/mecabrc'
        self.m = MeCab.Tagger(f'-d "/usr/lib/x86_64-linux-gnu/mecab/dic/mecab-ipadic-neologd"')

    def __call__(self, sentence):
        if "固有名詞" in self.m.parse(sentence).strip():
            return True
        return False

un = UniqueNounChecker()

#
wc("ラブホ"), un("ビットコイン")

(True, True)

In [17]:
# OCRのスキーマを定義
ocr_schema = pa.struct([
    ('content', pa.string()),
    ('det_score', pa.float32()),
    ('direction', pa.string()),
    ('points', pa.list_(pa.list_(pa.int32()))),
    ('rec_score', pa.float32())
])

# typeのスキーマを定義
type_schema = pa.map_(
    pa.string(), 
    pa.float32()
)

# お題画像のスキーマを定義
odai_schema = pa.schema([
    ('id', pa.int64()),
    ('date', pa.timestamp('s')),
    ('bokeCount', pa.int32()),
    ('category', pa.string()),
    ('largeUrl', pa.string()),
    ('imageOCR', pa.list_(ocr_schema)),  # OCR結果を可変長配列で格納
    ('imageCaption', pa.string()),
    ('imageType', type_schema)
])

# rateのスキーマを定義
rate_schema = pa.struct([
    ('date', pa.timestamp('s')),
    ('rate', pa.float32())
])

# ボケのスキーマを定義
boke_schema = pa.schema([
    ('id', pa.int64()),
    ('odaiId', pa.int64()),
    ('date', pa.timestamp('s')),
    ('text', pa.string()),
    ('isNG', pa.bool_()),
    ("isUniqueNoun", pa.bool_()),
    ('rateSum', pa.float32()),
    ('rateCount', pa.int32()),
    ('lastRateDate', pa.timestamp('s')),
    ('category', pa.string()),
    ('rates', pa.list_(rate_schema)) # 評価を可変長配列で格納
])

In [ ]:
BATCH_SIZE = 10000
ID_THRESHOLD = BATCH_SIZE
LAST_ODAI_ID = 9000000
odai_data_list = []
boke_data_list = []
image_data_list = []

for ODAI_ID in tqdm(range(0, LAST_ODAI_ID + 1)):

    # ---------------------------------------------------------
    # 4. バッチ保存処理（インデックスチェック）
    # ---------------------------------------------------------
    if ODAI_ID >= ID_THRESHOLD:
        ID_THRESHOLD += BATCH_SIZE

        if odai_data_list:
            odai_table = pa.Table.from_pylist(odai_data_list, schema=odai_schema)
            pq.write_table(odai_table, os.path.join(TARGET_ODAI_DIR, f"odai_{ODAI_ID // BATCH_SIZE}.parquet"))
        
        if boke_data_list:
            boke_table = pa.Table.from_pylist(boke_data_list, schema=boke_schema)
            pq.write_table(boke_table, os.path.join(TARGET_BOKE_DIR, f"boke_{ODAI_ID // BATCH_SIZE}.parquet"))

        for ID in image_data_list:
            os.makedirs(os.path.dirname(ID["target_path"]), exist_ok=True)
            shutil.copy2(ID["original_path"], ID["target_path"])
        
        odai_data_list = []
        boke_data_list = []
        image_data_list = []

    # ---------------------------------------------------------
    # 1. image 処理
    # ---------------------------------------------------------
    image_path = os.path.join(IMAGE_DIR, f"{ODAI_ID}.jpg")
    if os.path.exists(image_path):
        folder_id = (ODAI_ID // BATCH_SIZE) * BATCH_SIZE
        target_image_dir = os.path.join(TARGET_IMAGE_DIR, str(folder_id))
        target_image_path = os.path.join(target_image_dir, f"{ODAI_ID}.jpg")  

        if os.path.exists(target_image_path):
            continue

        if not os.path.exists(target_image_path):
            image_data_list.append({
                "original_path": image_path,
                "target_path": target_image_path
            })
        
    # ---------------------------------------------------------
    # 2. odai 処理
    # ---------------------------------------------------------
    odai_path = os.path.join(ODAI_DIR, f"{ODAI_ID}.json")
    if not os.path.exists(odai_path): 
        continue

    with open(odai_path, 'r') as f:
        odai = json.load(f)
    
    odai_date = odai.get('date')
    if odai_date is not None:
        odai_date = datetime.strptime(odai_date, "%Y-%m-%d %H:%M:%S")

    boke_count = odai.get('bokeCount')
    category = odai.get('category')
    photo = odai.get('photo')
    largeUrl = "https:" + photo.get("largeUrl") if photo else None

    # caption
    caption_path = os.path.join(CAPTION_DIR, f"{ODAI_ID}.json")
    if not os.path.exists(caption_path): 
        caption = None
    else:
        with open(caption_path, 'r') as f:
            caption = json.load(f)

    # ocr 
    ocr_path = os.path.join(OCR_DIR, f"{ODAI_ID}.json")
    if not os.path.exists(ocr_path):
        ocr = None  # OCR未実施状態
    else:
        with open(ocr_path, 'r') as f:
            ocr_data = json.load(f)[0]
            # 空配列 [] ならそのまま文字なし状態、データがあればその中身をセット
            ocr = ocr_data if (isinstance(ocr_data, list) and len(ocr_data) > 0) else []
        
    # type (Map型対応のための修正)
    type_path = os.path.join(TYPE_DIR, f"{ODAI_ID}.json")
    if not os.path.exists(type_path):
        image_type = None
    else:
        with open(type_path, 'r') as f:
            raw_type = json.load(f)
            # 辞書型を pa.map_ 用に (キー, バリュー) のリストタプルに変換
            image_type = list(raw_type.items()) if raw_type else None

    # お題リストに追加
    odai_data = {
        'id': ODAI_ID,
        'date': odai_date,
        'bokeCount': boke_count,
        'category': category,
        'largeUrl': largeUrl,
        'imageOCR': ocr,
        'imageCaption': caption,
        'imageType': image_type
    }
    odai_data_list.append(odai_data)

    # ---------------------------------------------------------
    # 3. boke 処理
    # ---------------------------------------------------------
    boke_path = os.path.join(BOKE_DIR, f"{ODAI_ID}.json")
    if not os.path.exists(boke_path): 
        # ボケが1件もない場合は、スキップせず bokes = [] として扱う
        bokes = []
    else:
        with open(boke_path, 'r') as f:
            bokes = json.load(f)

    for boke in bokes:
        boke_id = boke['id']
        text = boke.get('text')

        is_ng = wc(text)
        is_unique_noun = un(text)
        
        boke_date = boke.get('date')
        if boke_date is not None:
            boke_date = datetime.strptime(boke_date, "%Y-%m-%d %H:%M:%S")
            
        rateSum = boke.get('rateSum')
        rateCount = boke.get('rateCount')
        
        # lastRateDate の datetime 変換
        lastRateDate = boke.get('lastRateDate')
        if lastRateDate is not None:
            lastRateDate = datetime.strptime(lastRateDate, "%Y-%m-%d %H:%M:%S")
            
        odaiId = boke.get('odaiId')
        boke_category = boke.get('category')

        # rate
        rating_path = os.path.join(RATING_DIR, f"{boke_id}.json")
        if not os.path.exists(rating_path):
            rates = None

        else:
            with open(rating_path, 'r') as f:
                raw_rates = json.load(f)

            if "message" in raw_rates:
                rates = None

            else:
                rates = []
                for R in raw_rates:
                    r_date = R.get('date')  # 変数名競合を避けるため r_date に変更
                    if r_date is not None:
                        r_date = datetime.strptime(r_date, "%Y-%m-%d %H:%M:%S")
                    rate = R.get('rate')
                    rates.append({
                        "date": r_date,
                        "rate": rate
                    })
                
        boke_data = {
            "id": boke_id,
            "text": text,
            "isNG": is_ng,
            "isUniqueNoun": is_unique_noun,
            "date": boke_date,
            "rateSum": rateSum,
            "rateCount": rateCount,
            "lastRateDate": lastRateDate,
            "odaiId": odaiId,
            "category": boke_category,
            "rates": rates
        }
        boke_data_list.append(boke_data)


  1%|          | 93422/9000001 [11:48<13:42:15, 180.53it/s]